In [ ]:
from pathlib import Path
from collections import OrderedDict
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

LOG_DIR = Path(".")   # folder with FGSM*.txt files
OUT_DIR = Path("./project_plots")
OUT_DIR.mkdir(exist_ok=True, parents=True)

EPS_ORDER = [0.0078, 0.0156, 0.0313, 0.0627]
MODEL_ORDER = ["Dense", "Pruned 20%", "Pruned 40%", "Pruned 60%", "Pruned 80%"]

# Choose one epsilon for the "clean vs FGSM" bar chart
REPRESENTATIVE_EPS = 0.0313


# ============================================================
# OPTIONAL: pruning-region data from your earlier pruning logs
# Fill this only if you want the architecture/pruning plots too.
# If left as None, those plots are skipped.
# ============================================================

PRUNING_REGION_DATA = OrderedDict({
    "20%": OrderedDict({
        "features.0":   {"zeros": 12,       "total": 1728},
        "features.2":   {"zeros": 1081,     "total": 36864},
        "features.5":   {"zeros": 2259,     "total": 73728},
        "features.7":   {"zeros": 5153,     "total": 147456},
        "features.10":  {"zeros": 11888,    "total": 294912},
        "features.12":  {"zeros": 29155,    "total": 589824},
        "features.14":  {"zeros": 28558,    "total": 589824},
        "features.17":  {"zeros": 67819,    "total": 1179648},
        "features.19":  {"zeros": 169221,   "total": 2359296},
        "features.21":  {"zeros": 169379,   "total": 2359296},
        "features.24":  {"zeros": 158997,   "total": 2359296},
        "features.26":  {"zeros": 158191,   "total": 2359296},
        "features.28":  {"zeros": 177737,   "total": 2359296},
        "classifier.0": {"zeros": 23983254, "total": 102760448},
        "classifier.3": {"zeros": 1932216,  "total": 16777216},
        "classifier.6": {"zeros": 36626,    "total": 409600},
    }),
    "40%": OrderedDict({
        "features.0":   {"zeros": 22,       "total": 1728},
        "features.2":   {"zeros": 2274,     "total": 36864},
        "features.5":   {"zeros": 4644,     "total": 73728},
        "features.7":   {"zeros": 10741,    "total": 147456},
        "features.10":  {"zeros": 25295,    "total": 294912},
        "features.12":  {"zeros": 61231,    "total": 589824},
        "features.14":  {"zeros": 60025,    "total": 589824},
        "features.17":  {"zeros": 142424,   "total": 1179648},
        "features.19":  {"zeros": 355570,   "total": 2359296},
        "features.21":  {"zeros": 355758,   "total": 2359296},
        "features.24":  {"zeros": 333782,   "total": 2359296},
        "features.26":  {"zeros": 331546,   "total": 2359296},
        "features.28":  {"zeros": 371548,   "total": 2359296},
        "classifier.0": {"zeros": 47739751, "total": 102760448},
        "classifier.3": {"zeros": 3991886,  "total": 16777216},
        "classifier.6": {"zeros": 76594,    "total": 409600},
    }),
    "60%": OrderedDict({
        "features.0":   {"zeros": 38,       "total": 1728},
        "features.2":   {"zeros": 3696,     "total": 36864},
        "features.5":   {"zeros": 7634,     "total": 73728},
        "features.7":   {"zeros": 17970,    "total": 147456},
        "features.10":  {"zeros": 41997,    "total": 294912},
        "features.12":  {"zeros": 101518,   "total": 589824},
        "features.14":  {"zeros": 99560,    "total": 589824},
        "features.17":  {"zeros": 235240,   "total": 1179648},
        "features.19":  {"zeros": 586415,   "total": 2359296},
        "features.21":  {"zeros": 585848,   "total": 2359296},
        "features.24":  {"zeros": 549623,   "total": 2359296},
        "features.26":  {"zeros": 546885,   "total": 2359296},
        "features.28":  {"zeros": 609706,   "total": 2359296},
        "classifier.0": {"zeros": 70866010, "total": 102760448},
        "classifier.3": {"zeros": 6416247,  "total": 16777216},
        "classifier.6": {"zeros": 126250,   "total": 409600},
    }),
    "80%": OrderedDict({
        "features.0":   {"zeros": 70,       "total": 1728},
        "features.2":   {"zeros": 6043,     "total": 36864},
        "features.5":   {"zeros": 12701,    "total": 73728},
        "features.7":   {"zeros": 29419,    "total": 147456},
        "features.10":  {"zeros": 68735,    "total": 294912},
        "features.12":  {"zeros": 166162,   "total": 589824},
        "features.14":  {"zeros": 163186,   "total": 589824},
        "features.17":  {"zeros": 383213,   "total": 1179648},
        "features.19":  {"zeros": 945612,   "total": 2359296},
        "features.21":  {"zeros": 947222,   "total": 2359296},
        "features.24":  {"zeros": 888982,   "total": 2359296},
        "features.26":  {"zeros": 886160,   "total": 2359296},
        "features.28":  {"zeros": 974394,   "total": 2359296},
        "classifier.0": {"zeros": 92171477, "total": 102760448},
        "classifier.3": {"zeros": 9877909,  "total": 16777216},
        "classifier.6": {"zeros": 204897,   "total": 409600},
    }),
})

PRUNING_PERF = OrderedDict({
    "20%": {"pruning_only_val_acc": 73.30, "final_val_acc": 73.74},
    "40%": {"pruning_only_val_acc": 72.82, "final_val_acc": 73.66},
    "60%": {"pruning_only_val_acc": 72.12, "final_val_acc": 73.52},
    "80%": {"pruning_only_val_acc": 69.46, "final_val_acc": 72.90},
})


# ============================================================
# PARSING FGSM LOGS
# ============================================================

BLOCK_RE = re.compile(
    r"Epsilon:\s*([0-9.]+)\s*"
    r".*?examples_10_count:\s*(\d+)\s*"
    r".*?examples_01_count:\s*(\d+)\s*"
    r".*?examples_000_count:\s*(\d+)\s*"
    r".*?examples_001_count:\s*(\d+)\s*"
    r".*?examples_11_count:\s*(\d+)\s*"
    r".*?Clean accuracy:\s*([0-9.]+)\s*"
    r".*?FGSM accuracy\s*\(epsilon=.*?\):\s*([0-9.]+)",
    re.DOTALL
)

def parse_filename(path: Path):
    name = path.name

    m_dataset = re.search(r"_VGG_(10|100)\.txt$", name)
    if not m_dataset:
        raise ValueError(f"Could not infer dataset from filename: {name}")
    dataset = f"CIFAR-{m_dataset.group(1)}"

    if "FGSM_finetuned" in name:
        model_variant = "Dense"
    else:
        m_prune = re.search(r"prune_finetune_?(\d+)_VGG_", name)
        if not m_prune:
            raise ValueError(f"Could not infer pruning level from filename: {name}")
        model_variant = f"Pruned {int(m_prune.group(1))}%"

    return dataset, model_variant

def round_eps(x):
    return round(float(x) + 1e-12, 4)

def parse_all_logs(log_dir: Path):
    rows = []
    for path in sorted(log_dir.glob("FGSM*.txt")):
        dataset, model_variant = parse_filename(path)
        text = path.read_text(encoding="utf-8", errors="ignore")

        found = {}
        for m in BLOCK_RE.finditer(text):
            eps = round_eps(m.group(1))
            rec = {
                "dataset": dataset,
                "model_variant": model_variant,
                "epsilon": eps,
                "examples_10_count": int(m.group(2)),
                "examples_01_count": int(m.group(3)),
                "examples_000_count": int(m.group(4)),
                "examples_001_count": int(m.group(5)),
                "examples_11_count": int(m.group(6)),
                "clean_accuracy": float(m.group(7)),
                "fgsm_accuracy": float(m.group(8)),
                "source_file": path.name,
            }

            # deduplicate repeated blocks inside file
            if eps in found:
                prev = found[eps].copy()
                cur = rec.copy()
                prev.pop("source_file", None)
                cur.pop("source_file", None)
                if prev != cur:
                    raise ValueError(f"Inconsistent duplicate block in {path.name} at epsilon={eps}")
            else:
                found[eps] = rec

        if not found:
            raise ValueError(f"No epsilon blocks found in {path.name}")

        rows.extend(found.values())

    df = pd.DataFrame(rows)

    # derived rates
    for c in [
        "examples_10_count",
        "examples_01_count",
        "examples_000_count",
        "examples_001_count",
        "examples_11_count",
    ]:
        df[c.replace("_count", "_rate")] = df[c] / 10000.0

    df["accuracy_drop"] = df["clean_accuracy"] - df["fgsm_accuracy"]

    return df


# ============================================================
# HELPERS
# ============================================================

def save_fig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR / name, dpi=200, bbox_inches="tight")
    plt.close()

def ordered_sub(df, dataset):
    sub = df[df["dataset"] == dataset].copy()
    sub["model_variant"] = pd.Categorical(sub["model_variant"], categories=MODEL_ORDER, ordered=True)
    sub = sub.sort_values(["model_variant", "epsilon"])
    return sub

def pivot_metric(df, dataset, value_col):
    sub = ordered_sub(df, dataset)
    pt = sub.pivot_table(index="model_variant", columns="epsilon", values=value_col, aggfunc="first")
    cols = [e for e in EPS_ORDER if e in pt.columns]
    pt = pt[cols]
    return pt

def make_region_tables(pruning_data):
    REGIONS = OrderedDict({
        "Early conv": ["features.0", "features.2", "features.5", "features.7"],
        "Mid conv":   ["features.10", "features.12", "features.14"],
        "Late conv":  ["features.17", "features.19", "features.21", "features.24", "features.26", "features.28"],
        "Classifier": ["classifier.0", "classifier.3", "classifier.6"],
    })

    REGIONS_2WAY = OrderedDict({
        "All features": [k for k in pruning_data["20%"].keys() if k.startswith("features.")],
        "Classifier":   [k for k in pruning_data["20%"].keys() if k.startswith("classifier.")],
    })

    def weighted_region_sparsity(stats, layers):
        z = sum(stats[l]["zeros"] for l in layers)
        t = sum(stats[l]["total"] for l in layers)
        return z / t

    def unweighted_region_sparsity(stats, layers):
        vals = [stats[l]["zeros"] / stats[l]["total"] for l in layers]
        return sum(vals) / len(vals)

    def pruning_concentration(stats, layers):
        z_region = sum(stats[l]["zeros"] for l in layers)
        z_all = sum(v["zeros"] for v in stats.values())
        return z_region / z_all

    def region_param_share(stats, layers):
        t_region = sum(stats[l]["total"] for l in layers)
        t_all = sum(v["total"] for v in stats.values())
        return t_region / t_all

    def table(regions, fn):
        rows = []
        for region, layers in regions.items():
            row = {"Region": region}
            for k, stats in pruning_data.items():
                row[k] = fn(stats, layers)
            rows.append(row)
        return pd.DataFrame(rows)

    weighted_df = table(REGIONS, weighted_region_sparsity)
    unweighted_df = table(REGIONS, unweighted_region_sparsity)
    concentration_df = table(REGIONS, pruning_concentration)
    conv_vs_classifier_df = table(REGIONS_2WAY, weighted_region_sparsity)

    ref_stats = next(iter(pruning_data.values()))
    parameter_share_rows = []
    for region, layers in REGIONS.items():
        parameter_share_rows.append({
            "Region": region,
            "Parameter Share": region_param_share(ref_stats, layers)
        })
    parameter_share_df = pd.DataFrame(parameter_share_rows)

    return weighted_df, unweighted_df, concentration_df, conv_vs_classifier_df, parameter_share_df


# ============================================================
# PLOTS 1–8: FGSM / ERROR-TYPE PLOTS
# ============================================================

def plot_fgsm_accuracy_vs_epsilon(df, dataset):
    sub = ordered_sub(df, dataset)
    plt.figure(figsize=(7, 5))
    for model in MODEL_ORDER:
        s = sub[sub["model_variant"] == model]
        if len(s):
            plt.plot(s["epsilon"], s["fgsm_accuracy"], marker="o", label=model)
    plt.xlabel("Epsilon")
    plt.ylabel("FGSM Accuracy")
    plt.title(f"{dataset}: FGSM Accuracy vs Epsilon")
    plt.legend()
    save_fig(f"{dataset.lower().replace('-', '_')}_fgsm_accuracy_vs_epsilon.png")

def plot_clean_vs_fgsm_bar(df, dataset, epsilon=REPRESENTATIVE_EPS):
    sub = ordered_sub(df, dataset)
    s = sub[np.isclose(sub["epsilon"], epsilon)].copy()

    x = np.arange(len(s))
    width = 0.38

    plt.figure(figsize=(8, 5))
    plt.bar(x - width/2, s["clean_accuracy"], width=width, label="Clean Accuracy")
    plt.bar(x + width/2, s["fgsm_accuracy"], width=width, label="FGSM Accuracy")
    plt.xticks(x, s["model_variant"], rotation=20)
    plt.ylabel("Accuracy")
    plt.title(f"{dataset}: Clean vs FGSM Accuracy at epsilon={epsilon:.4f}")
    plt.legend()
    save_fig(f"{dataset.lower().replace('-', '_')}_clean_vs_fgsm_eps_{epsilon:.4f}.png")

def plot_accuracy_drop_vs_epsilon(df, dataset):
    sub = ordered_sub(df, dataset)
    plt.figure(figsize=(7, 5))
    for model in MODEL_ORDER:
        s = sub[sub["model_variant"] == model]
        if len(s):
            plt.plot(s["epsilon"], s["accuracy_drop"], marker="o", label=model)
    plt.xlabel("Epsilon")
    plt.ylabel("Clean Accuracy - FGSM Accuracy")
    plt.title(f"{dataset}: Accuracy Drop vs Epsilon")
    plt.legend()
    save_fig(f"{dataset.lower().replace('-', '_')}_accuracy_drop_vs_epsilon.png")

def plot_stacked_example_categories(df, dataset):
    sub = ordered_sub(df, dataset)

    categories = [
        "examples_10_rate",
        "examples_01_rate",
        "examples_000_rate",
        "examples_001_rate",
        "examples_11_rate",
    ]
    labels = ["10", "01", "000", "001", "11"]

    for model in MODEL_ORDER:
        s = sub[sub["model_variant"] == model].sort_values("epsilon")
        if s.empty:
            continue

        plt.figure(figsize=(8, 5))
        bottom = np.zeros(len(s))
        x = np.arange(len(s))

        for cat, label in zip(categories, labels):
            vals = s[cat].to_numpy()
            plt.bar(x, vals, bottom=bottom, label=label)
            bottom += vals

        plt.xticks(x, [f"{e:.4f}" for e in s["epsilon"]])
        plt.xlabel("Epsilon")
        plt.ylabel("Fraction of examples")
        plt.title(f"{dataset}: Example-category decomposition ({model})")
        plt.legend(title="Category", ncol=5)
        save_fig(f"{dataset.lower().replace('-', '_')}_{model.lower().replace(' ', '_').replace('%','')}_stacked_categories.png")

def plot_transition_10_vs_11(df, dataset):
    sub = ordered_sub(df, dataset)

    for model in MODEL_ORDER:
        s = sub[sub["model_variant"] == model].sort_values("epsilon")
        if s.empty:
            continue

        plt.figure(figsize=(7, 5))
        plt.plot(s["epsilon"], s["examples_10_rate"], marker="o", label="10 rate")
        plt.plot(s["epsilon"], s["examples_11_rate"], marker="o", label="11 rate")
        plt.xlabel("Epsilon")
        plt.ylabel("Fraction of examples")
        plt.title(f"{dataset}: 10 vs 11 transitions ({model})")
        plt.legend()
        save_fig(f"{dataset.lower().replace('-', '_')}_{model.lower().replace(' ', '_').replace('%','')}_10_vs_11.png")

def plot_heatmap(df, dataset, metric_col, title_suffix, out_name):
    pt = pivot_metric(df, dataset, metric_col)

    plt.figure(figsize=(6, 4.5))
    arr = pt.iloc[:, :].to_numpy()
    plt.imshow(arr, aspect="auto")

    plt.xticks(np.arange(len(pt.columns)), pt.columns)
    plt.yticks(np.arange(len(pt.index)), pt.index)
    plt.xlabel("Epsilon")
    plt.ylabel("Model Variant")
    plt.title(f"{dataset}: {title_suffix}")

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            plt.text(j, i, f"{arr[i, j]:.4f}", ha="center", va="center", fontsize=8)

    plt.colorbar()
    save_fig(f"{dataset.lower().replace('-', '_')}_{out_name}.png")

def plot_fixed_epsilon_bars(df, dataset):
    sub = ordered_sub(df, dataset)
    for eps in EPS_ORDER:
        s = sub[np.isclose(sub["epsilon"], eps)].copy()
        if s.empty:
            continue

        plt.figure(figsize=(7, 5))
        x = np.arange(len(s))
        plt.bar(x, s["fgsm_accuracy"])
        plt.xticks(x, s["model_variant"], rotation=20)
        plt.ylabel("FGSM Accuracy")
        plt.title(f"{dataset}: FGSM Accuracy across pruning levels at epsilon={eps:.4f}")
        save_fig(f"{dataset.lower().replace('-', '_')}_fixed_eps_{eps:.4f}_bars.png")

def plot_cifar10_vs_cifar100(df, model_variant):
    sub = df[df["model_variant"] == model_variant].copy()

    plt.figure(figsize=(7, 5))
    for dataset in ["CIFAR-10", "CIFAR-100"]:
        s = sub[sub["dataset"] == dataset].sort_values("epsilon")
        if len(s):
            plt.plot(s["epsilon"], s["fgsm_accuracy"], marker="o", label=dataset)

    plt.xlabel("Epsilon")
    plt.ylabel("FGSM Accuracy")
    plt.title(f"{model_variant}: CIFAR-10 vs CIFAR-100")
    plt.legend()
    save_fig(f"{model_variant.lower().replace(' ', '_').replace('%','')}_cifar10_vs_cifar100.png")


# ============================================================
# PLOTS 9–12: PRUNING / ARCHITECTURE PLOTS
# ============================================================

def plot_weighted_region_sparsity(weighted_df):
    plt.figure(figsize=(8, 5))
    regions = weighted_df["Region"].tolist()
    x = np.arange(len(regions))
    width = 0.18

    for i, col in enumerate(["20%", "40%", "60%", "80%"]):
        plt.bar(x + (i - 1.5) * width, weighted_df[col], width=width, label=col)

    plt.xticks(x, regions, rotation=20)
    plt.ylabel("Weighted region sparsity")
    plt.title("Weighted pruning by model region")
    plt.legend(title="Global pruning")
    save_fig("weighted_pruning_by_region.png")

def plot_conv_vs_classifier(conv_vs_classifier_df):
    plt.figure(figsize=(7, 5))
    regions = conv_vs_classifier_df["Region"].tolist()
    x = np.arange(len(regions))
    width = 0.18

    for i, col in enumerate(["20%", "40%", "60%", "80%"]):
        plt.bar(x + (i - 1.5) * width, conv_vs_classifier_df[col], width=width, label=col)

    plt.xticks(x, regions)
    plt.ylabel("Weighted sparsity")
    plt.title("Conv vs Classifier sparsity")
    plt.legend(title="Global pruning")
    save_fig("conv_vs_classifier_sparsity.png")

def plot_pruning_concentration(concentration_df):
    plt.figure(figsize=(8, 5))
    regions = concentration_df["Region"].tolist()
    x = np.arange(len(regions))
    width = 0.18

    for i, col in enumerate(["20%", "40%", "60%", "80%"]):
        plt.bar(x + (i - 1.5) * width, concentration_df[col], width=width, label=col)

    plt.xticks(x, regions, rotation=20)
    plt.ylabel("Share of all pruned weights")
    plt.title("Pruning concentration by region")
    plt.legend(title="Global pruning")
    save_fig("pruning_concentration_by_region.png")

def plot_parameter_share(parameter_share_df):
    plt.figure(figsize=(7, 5))
    x = np.arange(len(parameter_share_df))
    plt.bar(x, parameter_share_df["Parameter Share"])
    plt.xticks(x, parameter_share_df["Region"], rotation=20)
    plt.ylabel("Parameter share")
    plt.title("Parameter share of each region")
    save_fig("parameter_share_by_region.png")

def plot_fine_tuning_recovery(perf):
    levels = list(perf.keys())
    before = [perf[k]["pruning_only_val_acc"] for k in levels]
    after = [perf[k]["final_val_acc"] for k in levels]
    recovery = [a - b for a, b in zip(after, before)]

    x = np.arange(len(levels))
    width = 0.38

    plt.figure(figsize=(7, 5))
    plt.bar(x - width/2, before, width=width, label="After pruning")
    plt.bar(x + width/2, after, width=width, label="After fine-tuning")
    plt.xticks(x, levels)
    plt.ylabel("Validation Accuracy (%)")
    plt.title("Fine-tuning recovery after pruning")
    plt.legend()
    save_fig("fine_tuning_recovery.png")

    plt.figure(figsize=(7, 5))
    plt.bar(x, recovery)
    plt.xticks(x, levels)
    plt.ylabel("Recovery (percentage points)")
    plt.title("Validation accuracy recovery due to fine-tuning")
    save_fig("fine_tuning_recovery_gain.png")


# ============================================================
# DRIVER
# ============================================================

def main():
    df = parse_all_logs(LOG_DIR)

    df.to_csv(OUT_DIR / "parsed_fgsm_records.csv", index=False)

    # ---- CIFAR-10 plots ----
    for dataset in ["CIFAR-10", "CIFAR-100"]:
        plot_fgsm_accuracy_vs_epsilon(df, dataset)
        plot_clean_vs_fgsm_bar(df, dataset, epsilon=REPRESENTATIVE_EPS)
        plot_accuracy_drop_vs_epsilon(df, dataset)
        plot_stacked_example_categories(df, dataset)
        plot_transition_10_vs_11(df, dataset)
        plot_heatmap(df, dataset, "fgsm_accuracy", "FGSM Accuracy Heatmap", "fgsm_accuracy_heatmap")
        plot_heatmap(df, dataset, "examples_10_rate", "10 Rate Heatmap", "examples_10_heatmap")
        plot_heatmap(df, dataset, "examples_11_rate", "11 Rate Heatmap", "examples_11_heatmap")
        plot_fixed_epsilon_bars(df, dataset)

    # ---- CIFAR-10 vs CIFAR-100 comparison ----
    for model in MODEL_ORDER:
        plot_cifar10_vs_cifar100(df, model)

    # ---- Pruning region plots ----
    if PRUNING_REGION_DATA is not None:
        weighted_df, unweighted_df, concentration_df, conv_vs_classifier_df, parameter_share_df = make_region_tables(PRUNING_REGION_DATA)

        weighted_df.to_csv(OUT_DIR / "weighted_region_sparsity.csv", index=False)
        unweighted_df.to_csv(OUT_DIR / "unweighted_region_sparsity.csv", index=False)
        concentration_df.to_csv(OUT_DIR / "pruning_concentration.csv", index=False)
        conv_vs_classifier_df.to_csv(OUT_DIR / "conv_vs_classifier.csv", index=False)
        parameter_share_df.to_csv(OUT_DIR / "parameter_share.csv", index=False)

        plot_weighted_region_sparsity(weighted_df)
        plot_conv_vs_classifier(conv_vs_classifier_df)
        plot_pruning_concentration(concentration_df)
        plot_parameter_share(parameter_share_df)

    # ---- Fine-tuning recovery ----
    if PRUNING_PERF is not None:
        plot_fine_tuning_recovery(PRUNING_PERF)

    print(f"Saved plots to: {OUT_DIR.resolve()}")

if __name__ == "__main__":
    main()

In [ ]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# =========================
# Loading + parsing helpers
# =========================

def load_tables(json_path: str) -> List[Dict[str, Any]]:
    """
    Load the list of table dictionaries from a JSON file.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_title(title: str) -> str:
    """
    Normalize a title for easier matching.
    """
    return re.sub(r"\s+", " ", title.strip().lower())


def find_table(tables: List[Dict[str, Any]], title_keyword: str) -> Dict[str, Any]:
    """
    Find the first table whose title contains the given keyword.
    Case-insensitive partial match.
    """
    key = normalize_title(title_keyword)
    for table in tables:
        if key in normalize_title(table["title"]):
            return table
    raise ValueError(f"No table found with keyword: {title_keyword}")


def parse_value(x):
    """
    Convert strings like '92.24%', '0.7263', '20%' to float.
    Keeps raw strings if conversion fails.
    """
    if isinstance(x, (int, float)):
        return float(x)

    if not isinstance(x, str):
        return x

    x = x.strip()
    if x.endswith("%"):
        try:
            return float(x[:-1])
        except ValueError:
            return x

    try:
        return float(x)
    except ValueError:
        return x


def table_to_dataframe(table: Dict[str, Any], percent_to_fraction: bool = False) -> pd.DataFrame:
    """
    Convert a JSON table into a pandas DataFrame.
    If percent_to_fraction=True, values like '92.24%' become 0.9224.
    Otherwise they become 92.24.
    """
    df = pd.DataFrame(table["data"], columns=table["headers"])

    for col in df.columns[1:]:
        converted = df[col].apply(parse_value)
        if percent_to_fraction:
            converted = converted.apply(
                lambda v: v / 100.0 if isinstance(v, (int, float)) and "%" in str(df[col].iloc[0]) else v
            )
        df[col] = converted

    return df


def melt_metric_table(
    df: pd.DataFrame,
    id_col: str,
    var_name: str = "epsilon",
    value_name: str = "value"
) -> pd.DataFrame:
    """
    Convert a wide table like:
      Model | 0.0078 | 0.0156 | 0.0313
    into long form.
    """
    out = df.melt(id_vars=[id_col], var_name=var_name, value_name=value_name)
    out[var_name] = out[var_name].astype(str).str.extract(r'([0-9.]+)').astype(float)
    return out


# =========================
# Generic plotting helpers
# =========================

def save_or_show(output_path: Optional[str] = None):
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_grouped_bar(
    df: pd.DataFrame,
    x_col: str,
    value_cols: List[str],
    title: str,
    xlabel: str,
    ylabel: str,
    rotation: int = 0,
    figsize=(10, 5),
    output_path: Optional[str] = None
):
    """
    Grouped bar chart for one categorical axis and multiple numeric columns.
    """
    x = np.arange(len(df[x_col]))
    width = 0.8 / len(value_cols)

    plt.figure(figsize=figsize)
    for i, col in enumerate(value_cols):
        plt.bar(x + i * width - (len(value_cols) - 1) * width / 2, df[col], width=width, label=col)

    plt.xticks(x, df[x_col], rotation=rotation)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    save_or_show(output_path)


def plot_line_by_group(
    long_df: pd.DataFrame,
    group_col: str,
    x_col: str,
    y_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
    marker: str = "o",
    figsize=(10, 5),
    output_path: Optional[str] = None
):
    """
    Line chart for long-form data.
    """
    plt.figure(figsize=figsize)

    for group_name, subdf in long_df.groupby(group_col):
        subdf = subdf.sort_values(x_col)
        plt.plot(subdf[x_col], subdf[y_col], marker=marker, label=group_name)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, alpha=0.3)
    save_or_show(output_path)


def plot_horizontal_bar(
    df: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    xlabel: str,
    figsize=(8, 5),
    output_path: Optional[str] = None
):
    """
    Horizontal bar chart for a single metric.
    """
    plt.figure(figsize=figsize)
    plt.barh(df[label_col], df[value_col])
    plt.title(title)
    plt.xlabel(xlabel)
    save_or_show(output_path)


def plot_stacked_bar(
    df: pd.DataFrame,
    x_col: str,
    stack_cols: List[str],
    title: str,
    xlabel: str,
    ylabel: str,
    rotation: int = 0,
    figsize=(10, 6),
    output_path: Optional[str] = None
):
    """
    Stacked bar chart.
    """
    plt.figure(figsize=figsize)
    bottom = np.zeros(len(df))

    for col in stack_cols:
        plt.bar(df[x_col], df[col], bottom=bottom, label=col)
        bottom += df[col].values

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation)
    plt.legend()
    save_or_show(output_path)


def plot_heatmap(
    df: pd.DataFrame,
    index_col: str,
    title: str,
    figsize=(8, 5),
    cmap: str = "viridis",
    output_path: Optional[str] = None
):
    """
    Heatmap from a wide DataFrame. The first column is treated as row labels.
    """
    heat_df = df.set_index(index_col)

    plt.figure(figsize=figsize)
    plt.imshow(heat_df.values, aspect="auto", cmap=cmap)
    plt.colorbar(label="Value")
    plt.xticks(range(len(heat_df.columns)), heat_df.columns, rotation=45, ha="right")
    plt.yticks(range(len(heat_df.index)), heat_df.index)
    plt.title(title)
    save_or_show(output_path)


# =========================
# Task-specific wrappers
# =========================

def plot_clean_accuracy(tables: List[Dict[str, Any]], dataset_name: str):
    table = find_table(tables, "Clean Accuracy by Model")
    df = table_to_dataframe(table)

    plot_grouped_bar(
        df=df,
        x_col=df.columns[0],
        value_cols=[df.columns[1]],
        title=f"{dataset_name}: Clean Accuracy by Model",
        xlabel="Model",
        ylabel="Accuracy"
    )


def plot_adv_accuracy_initial_subset(tables: List[Dict[str, Any]], dataset_name: str):
    table = find_table(tables, "Adversarial accuracy on initially-correct subset")
    df = table_to_dataframe(table)
    long_df = melt_metric_table(df, id_col=df.columns[0], var_name="epsilon", value_name="adv_acc")

    plot_line_by_group(
        long_df=long_df,
        group_col=df.columns[0],
        x_col="epsilon",
        y_col="adv_acc",
        title=f"{dataset_name}: FGSM Adversarial Accuracy on Initially-Correct Subset",
        xlabel="Epsilon",
        ylabel="Adversarial Accuracy"
    )


def plot_attack_success_rate(tables: List[Dict[str, Any]], dataset_name: str):
    table = find_table(tables, "Attack success rate")
    df = table_to_dataframe(table)
    long_df = melt_metric_table(df, id_col=df.columns[0], var_name="epsilon", value_name="asr")

    plot_line_by_group(
        long_df=long_df,
        group_col=df.columns[0],
        x_col="epsilon",
        y_col="asr",
        title=f"{dataset_name}: FGSM Attack Success Rate",
        xlabel="Epsilon",
        ylabel="ASR"
    )


def plot_transfer_heatmap(tables: List[Dict[str, Any]], title_keyword: str, dataset_name: str):
    table = find_table(tables, title_keyword)
    df = table_to_dataframe(table)

    plot_heatmap(
        df=df,
        index_col=df.columns[0],
        title=f"{dataset_name}: {table['title']}"
    )


def plot_weighted_region_sparsity(tables: List[Dict[str, Any]], dataset_name: str):
    table = find_table(tables, "Weighted")
    df = table_to_dataframe(table)

    plot_grouped_bar(
        df=df,
        x_col=df.columns[0],
        value_cols=list(df.columns[1:]),
        title=f"{dataset_name}: Weighted Pruning by Model Region",
        xlabel="Region",
        ylabel="Sparsity (%)"
    )


def plot_parameter_share(tables: List[Dict[str, Any]], dataset_name: str):
    table = find_table(tables, "Parameter share")
    df = table_to_dataframe(table)

    plot_horizontal_bar(
        df=df,
        label_col=df.columns[0],
        value_col=df.columns[1],
        title=f"{dataset_name}: Parameter Share by Region",
        xlabel="Share of Total Parameters (%)"
    )


def plot_example_breakdown_for_model(
    tables: List[Dict[str, Any]],
    title_keyword: str,
    dataset_name: str
):
    table = find_table(tables, title_keyword)
    df = table_to_dataframe(table)

    # choose epsilon column name automatically
    epsilon_col = "epsilon"
    state_cols = [c for c in df.columns if c.startswith("examples_")]

    df["epsilon"] = df["epsilon"].astype(str)

    plot_stacked_bar(
        df=df,
        x_col=epsilon_col,
        stack_cols=state_cols,
        title=f"{dataset_name}: {table['title']}",
        xlabel="Epsilon",
        ylabel="Count"
    )


# =========================
# Batch plotting
# =========================

def run_core_plots(json_path: str, dataset_name: str):
    """
    Generate the most useful plots for a report.
    """
    tables = load_tables(json_path)

    plot_clean_accuracy(tables, dataset_name)
    plot_adv_accuracy_initial_subset(tables, dataset_name)
    plot_attack_success_rate(tables, dataset_name)
    plot_transfer_heatmap(tables, "Transferability: Dense → Pruned", dataset_name)
    plot_transfer_heatmap(tables, "Transferability: Pruned → Dense", dataset_name)
    plot_weighted_region_sparsity(tables, dataset_name)
    plot_parameter_share(tables, dataset_name)

    # example breakdowns
    # adapt title keywords to your two files
    for keyword in [
        "FGSM examples breakdown for Dense model",
        "FGSM examples breakdown for Pruned 20%",
        "FGSM examples breakdown for Pruned 40%",
        "FGSM examples breakdown for Pruned 60%",
        "FGSM examples breakdown for Pruned 80%",
        "Example Counts - Dense Model",
        "Example Counts - Pruned 20%",
        "Example Counts - Pruned 40%",
        "Example Counts - Pruned 60%",
        "Example Counts - Pruned 80%",
    ]:
        try:
            plot_example_breakdown_for_model(tables, keyword, dataset_name)
        except ValueError:
            pass